# Remotion Colab Render Workflow
這份筆記本專為您的 Remotion 專案設計，支援：
- 自動掛載 Google Drive
- 修正 Colab 上 `eslint-config-remotion` 的依賴衝突
- 自動建立所需的素材資料夾 (`drive_assets`)
- 輸出相容 Windows Media Player 的 yuv420p 影片
- 自動複製成果至 Google Drive

In [2]:
# Cell 0: 掛載 Drive 與 Clone 專案
from google.colab import drive
import os

drive.mount('/content/drive')

if not os.path.exists('/content/remotion'):
    !git clone https://github.com/Allen930311/remotion.git /content/remotion
else:
    %cd /content/remotion
    !git pull origin main

%cd /content/remotion

# 確保 drive_assets 選項目錄存在 (根據您的 auto-skill 經驗回報)
drive_assets_path = '/content/drive/MyDrive/remotion_assets'
os.makedirs(drive_assets_path, exist_ok=True)
os.makedirs(os.path.join(drive_assets_path, '02_Audio'), exist_ok=True)
os.makedirs(os.path.join(drive_assets_path, '03_Images'), exist_ok=True)
print("環境與專案取得完成！")

Mounted at /content/drive
Cloning into '/content/remotion'...
remote: Enumerating objects: 165, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (125/125), done.
remote: Total 165 (delta 26), reused 160 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (165/165), 156.00 KiB | 2.74 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/remotion
環境與專案取得完成！


In [3]:
# Cell 1: 依賴修復與安裝
%cd /content/remotion
import json

package_json_path = 'package.json'
with open(package_json_path, 'r', encoding='utf-8') as f:
    pkg = json.load(f)

# 移除 eslint-config-remotion 避免在 Colab 報錯
if 'devDependencies' in pkg and 'eslint-config-remotion' in pkg['devDependencies']:
    del pkg['devDependencies']['eslint-config-remotion']
    with open(package_json_path, 'w', encoding='utf-8') as f:
        json.dump(pkg, f, indent=2)
    print("[依賴修復] 已移除 eslint-config-remotion")

!npm install

/content/remotion
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹npm warn deprecated rimraf@3.0.2: Rimraf versions prior to v4 are no longer supported
⠹⠸⠼⠴⠦npm warn deprecated glob@7.2.3: Old versions of glob are not supported, and contain widely publicized security vulnerabilities, which have been fixed in the current version. Please update. Support for old versions may be purchased (at exorbitant rates) by contacting i@izs.me
⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧npm warn deprecated @humanwhocodes/config-array@0.13.0: Use @eslint/config-array instead
⠧npm warn deprecated @humanwhocodes/object-schema@2.0.3: Use @eslint/object-schema instead
⠧⠇npm warn deprecated inflight@1.0.6: This module is not supported, and leaks memory. Do not use it. Check out lru-cache if you want a good and tested way to coalesce async requests by a key value, which is much more comprehensive and powerful.
⠇⠏⠋⠙npm warn deprecated source-map@0.8.0-beta.0: The work that was done in this beta branch won't be included in future versions
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧

In [4]:
# Cell 2: 渲染設定與執行
%cd /content/remotion
import os
import shutil

# 設定環境變數 (如需更改參數請修改此處)
os.environ["ENTRY_FILE"] = "projects/colab-showcase/src/index.tsx"  # Remotion 進入點檔案
os.environ["COMP_NAME"] = "ColabShowcase"         # Composition 名稱
os.environ["OUTPUT_FILE"] = "out/video.mp4"

# 在 Colab 渲染時，建議使用 npx remotion render 搭配相容性參數
# 確保影片能在 Windows 預設播放器正常播放 (-pixel-format yuv420p, -codec h264)
render_cmd = f"npx remotion render {os.environ['ENTRY_FILE']} {os.environ['COMP_NAME']} {os.environ['OUTPUT_FILE']} --pixel-format=yuv420p --codec=h264 --concurrency=2"
print(f"Executing: {render_cmd}")

!{render_cmd}

# 將成品複製到 Google Drive
output_path = f"/content/remotion/{os.environ['OUTPUT_FILE']}"
drive_dest = f"/content/drive/MyDrive/remotion_assets/video_rendered.mp4"

if os.path.exists(output_path):
    shutil.copy(output_path, drive_dest)
    print(f"\n✅ 影片已成功輸出並複製至 Drive: {drive_dest}")
else:
    print("\n❌ 找不到輸出的影片，渲染可能失敗。")

/content/remotion
Executing: npx remotion render src/index.ts MyComp out/video.mp4 --pixel-format=yuv420p --codec=h264
⠙
Error: concurrency is set higher than the amount of CPU cores available. Available CPU cores: 2, value set: 8
    at Object.validateConcurrency (/content/remotion/node_modules/@remotion/renderer/dist/validate-concurrency.js:24:23)
    at renderVideoFlow (/content/remotion/node_modules/@remotion/cli/dist/render-flows/render.js:67:32)
    at render (/content/remotion/node_modules/@remotion/cli/dist/render.js:168:40)
    at cli (/content/remotion/node_modules/@remotion/cli/dist/index.js:98:39)
    at process.processTicksAndRejections (node:internal/process/task_queues:95:5)
⠙
❌ 找不到輸出的影片，渲染可能失敗。
